# NB7 — Fine-tuning d'un transformer

Notebook du pipeline **P25**.

## Portée du notebook

Ce notebook réalise un **fine-tuning complet** d'un transformer pour la classification binaire de tweets catastrophiques.

- `train.csv` et `test.csv` sont supposés déjà préparés dans `../../data/` ;
- un **jeu de validation stratifié** est construit à partir du train ;
- `test.csv` est utilisé uniquement pour l'évaluation finale.

Le modèle par défaut est `distilbert-base-uncased` pour rester portable.

In [4]:
!pip install datasets


[notice] A new release of pip is available: 25.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# Installation éventuelle (décommente si nécessaire)
# !pip install pandas numpy scikit-learn transformers datasets accelerate evaluate torch openpyxl

import os
import json
import numpy as np
import pandas as pd

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import evaluate

from nlp_disaster_utils import (
    seed_everything,
    load_train_test_xy,
    stratified_validation_split,
    classification_metrics_from_predictions,
    round_results,
    metric_matrix_from_results,
    save_results_bundle,
)

seed_everything(42)

ImportError: The pyarrow installation is not built with support for 'dataset' (DLL load failed while importing _dataset: Le module spécifié est introuvable.)

In [ ]:
DATA_DIR = "../../data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TEST_PATH = f"{DATA_DIR}/test.csv"

TEXT_COL = "text"
LABEL_COL = "target"
USE_AUX_TEXT_COLUMNS = False
LOWERCASE_TEXT = False

VAL_SIZE_WITHIN_TRAIN = 0.10
RANDOM_STATE = 42

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 96
BATCH_SIZE = 16
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

OUTPUT_STEM = "NB7_transformer_finetuning"
OUTPUT_DIR = "./hf_outputs"
RESULTS_DIR = "results"

In [ ]:
df_train, X_train_full, y_train_full, df_test, X_test, y_test = load_train_test_xy(
    train_path=TRAIN_PATH,
    test_path=TEST_PATH,
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    use_extra_cols=USE_AUX_TEXT_COLUMNS,
    lowercase=LOWERCASE_TEXT,
)

X_train, X_val, y_train, y_val = stratified_validation_split(
    X_train_full,
    y_train_full,
    val_size=VAL_SIZE_WITHIN_TRAIN,
    random_state=RANDOM_STATE,
)

train_ds = Dataset.from_dict({"text": list(X_train), "label": list(map(int, y_train))})
val_ds = Dataset.from_dict({"text": list(X_val), "label": list(map(int, y_val))})
test_ds = Dataset.from_dict({"text": list(X_test), "label": list(map(int, y_test))})

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

train_ds = train_ds.map(tokenize_batch, batched=True)
val_ds = val_ds.map(tokenize_batch, batched=True)
test_ds = test_ds.map(tokenize_batch, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

In [ ]:
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision_metric.compute(predictions=preds, references=labels, average="binary")["precision"],
        "recall": recall_metric.compute(predictions=preds, references=labels, average="binary")["recall"],
        "f1": f1_metric.compute(predictions=preds, references=labels, average="binary")["f1"],
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
    save_total_limit=1,
    seed=42,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
def softmax(logits):
    exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)

train_pred_output = trainer.predict(train_ds)
test_pred_output = trainer.predict(test_ds)

train_logits = train_pred_output.predictions
test_logits = test_pred_output.predictions

train_scores = softmax(train_logits)[:, 1]
test_scores = softmax(test_logits)[:, 1]

train_preds = np.argmax(train_logits, axis=1)
test_preds = np.argmax(test_logits, axis=1)

result = {"pipeline": "P25_Finetuned_DistilBERT"}
result.update(classification_metrics_from_predictions(np.array(y_train), train_preds, train_scores, prefix="train"))
result.update(classification_metrics_from_predictions(np.array(y_test), test_preds, test_scores, prefix="test"))

results_df = round_results(pd.DataFrame([result]))
results_df

In [ ]:
display(results_df)
metric_view = round_results(metric_matrix_from_results(results_df))
display(metric_view)

save_results_bundle(results_df, output_dir=RESULTS_DIR, stem=OUTPUT_STEM)
print(f"Fichiers CSV/XLSX enregistrés dans ./{RESULTS_DIR}")